Librerias

In [1]:
%pip install -q kaggle
%pip install -q pillow
%pip install -q pandas
%pip install -q matplotlib
%pip install -q numpy
%pip install imagehash --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
%pip install -q kagglehub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import os
import imagehash
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from procesamiento import dividir_train_val_test, calcular_hash, marcar_duplicado_verificado, son_realmente_duplicados
from sklearn.preprocessing import LabelEncoder




Cosas necesarias para tener el  train test

In [4]:
N_POR_CLASE = 600
SEMILLA = 42
TAM_OBJETIVO = (64, 64)
np.random.seed(SEMILLA)


RUTA_DATASET = r"archive/asl_alphabet_train/asl_alphabet_train"

datos_img = []

for clase in sorted(os.listdir(RUTA_DATASET)):
    carpeta = os.path.join(RUTA_DATASET, clase)
    if not os.path.isdir(carpeta):
        continue

    for archivo in os.listdir(carpeta):
        ruta = os.path.join(carpeta, archivo)
        try:
            imagen = Image.open(ruta)
            ancho, alto = imagen.size
            datos_img.append({
                "clase": clase,
                "archivo": archivo,
                "ruta": ruta,
                "ancho": ancho,
                "alto": alto,
                "modo": imagen.mode,
                "formato": imagen.format
            })
        except Exception as e:
            print(f"No se pudo abrir {ruta}")

datos_img = pd.DataFrame(datos_img)

print("Cantidad total de imágenes:")
print(len(datos_img))

datos_img.head()

submuestra = (
    datos_img.groupby("clase", group_keys=False)
    .apply(lambda x: x.sample(n=min(N_POR_CLASE, len(x)), random_state=SEMILLA))
    .reset_index(drop=True)
)

    
submuestra["phash"] = submuestra["ruta"].apply(lambda r: calcular_hash(r, hash_size=16))


Cantidad total de imágenes:
87000


C:\Users\belen\AppData\Local\Temp\ipykernel_53544\867506301.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(N_POR_CLASE, len(x)), random_state=SEMILLA))


duplicados y transformaciones

In [5]:
def cargar_y_preprocesar(df, tam=TAM_OBJETIVO):
    """Carga imágenes desde 'ruta', redimensiona y normaliza a [0, 1]."""
    imagenes = []
    for ruta in df["ruta"]:
        img = Image.open(ruta).convert("RGB").resize(tam)
        arr = np.array(img, dtype=np.float32) / 255.0
        imagenes.append(arr)
    X = np.stack(imagenes)
    y = df["clase"].values
    return X, y

duplicados = submuestra[submuestra.duplicated(subset="phash", keep=False)]
print(f"\nImágenes con hash repetido: {len(duplicados)} de {len(submuestra)}")

indices_duplicados_reales = []
for phash_val, grupo in duplicados.groupby("phash"):
    indices_duplicados_reales.extend(marcar_duplicado_verificado(grupo))

submuestra_limpia = submuestra.drop(index=indices_duplicados_reales).reset_index(drop=True)
print(f"Duplicados confirmados y eliminados: {len(indices_duplicados_reales)}")
print(f"Submuestra final: {len(submuestra_limpia)} imágenes")

# Verificar que la limpieza no haya desbalanceado alguna clase
print("\nDistribución de clases tras limpieza:")
print(submuestra_limpia["clase"].value_counts().sort_values())


#SPLIT TRAIN / VAL / TEST (70/15/15, estratificado)
train_df, val_df, test_df = dividir_train_val_test(
    df=submuestra_limpia,
    col_clase="clase",
    semilla=SEMILLA
)


# PREPROCESAMIENTO FINAL: resize + normalización -> arrays listos para modelar
X_train, y_train_raw = cargar_y_preprocesar(train_df)
X_val, y_val_raw = cargar_y_preprocesar(val_df)
X_test, y_test_raw = cargar_y_preprocesar(test_df)

# Codificación de etiquetas (texto -> enteros); se ajusta SOLO con train
codificador = LabelEncoder()
codificador.fit(y_train_raw)

y_train = codificador.transform(y_train_raw)
y_val = codificador.transform(y_val_raw)
y_test = codificador.transform(y_test_raw)

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print(f"Rango de X_train tras normalizar: [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"Clases codificadas: {dict(zip(codificador.classes_, range(len(codificador.classes_))))}")



Imágenes con hash repetido: 22 de 17400
Duplicados confirmados y eliminados: 9
Submuestra final: 17391 imágenes

Distribución de clases tras limpieza:
clase
E          599
U          599
O          599
Y          599
B          599
M          599
K          599
space      599
I          599
A          600
D          600
C          600
F          600
L          600
J          600
G          600
X          600
del        600
nothing    600
N          600
P          600
Q          600
R          600
T          600
H          600
W          600
S          600
V          600
Z          600
Name: count, dtype: int64
Train: 12173 (70.0%) | Val: 2609 (15.0%) | Test: 2609 (15.0%)

X_train: (12173, 64, 64, 3), y_train: (12173,)
X_val:   (2609, 64, 64, 3), y_val:   (2609,)
X_test:  (2609, 64, 64, 3), y_test:  (2609,)
Rango de X_train tras normalizar: [0.00, 1.00]
Clases codificadas: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 1

In [6]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [7]:


print("Forma original de los datos:")
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)


X_train_svm = X_train.reshape(X_train.shape[0], -1)
X_val_svm = X_val.reshape(X_val.shape[0], -1)
X_test_svm = X_test.reshape(X_test.shape[0], -1)

print("\nForma después de aplanar:")
print("X_train_svm:", X_train_svm.shape)
print("X_val_svm:  ", X_val_svm.shape)
print("X_test_svm: ", X_test_svm.shape ) 

print("\nRango de los datos:")
print( f"[{X_train_svm.min():.2f}, "f"{X_train_svm.max():.2f}]"
)

Forma original de los datos:
X_train: (12173, 64, 64, 3)
X_val:   (2609, 64, 64, 3)
X_test:  (2609, 64, 64, 3)

Forma después de aplanar:
X_train_svm: (12173, 12288)
X_val_svm:   (2609, 12288)
X_test_svm:  (2609, 12288)

Rango de los datos:
[0.00, 1.00]


In [8]:


configuraciones_svm = [
    {"nombre": "SVM C=0.01", "C": 0.01},
    {"nombre": "SVM C=0.1", "C": 0.1}
]

print("configuraciones que serán evaluadas:")

for configuracion in configuraciones_svm:
    print(configuracion)

configuraciones que serán evaluadas:
{'nombre': 'SVM C=0.01', 'C': 0.01}
{'nombre': 'SVM C=0.1', 'C': 0.1}


In [10]:
N_TRAIN_SVM = 5000

rng = np.random.default_rng(SEMILLA)

indices_svm = rng.choice(
    len(X_train_svm),
    size=N_TRAIN_SVM,
    replace=False
)

X_train_svm_reducido = X_train_svm[indices_svm]
y_train_svm_reducido = y_train[indices_svm]

print("Train original:", X_train_svm.shape)
print("Train usado para SVM:", X_train_svm_reducido.shape)

Train original: (12173, 12288)
Train usado para SVM: (5000, 12288)


In [12]:
resultados_svm = []
modelos_svm = {}

for configuracion in configuraciones_svm:

    nombre = configuracion["nombre"]
    valor_c = configuracion["C"]

    print("\n" + "=" * 50)
    print(f"Entrenando {nombre}")
    print("=" * 50)

    modelo = LinearSVC(
        C=valor_c,
        random_state=SEMILLA,
        max_iter=3000,
        dual="auto"
    )

    inicio = time.time()

    modelo.fit(
        X_train_svm_reducido,
        y_train_svm_reducido
    )

    tiempo = time.time() - inicio

    pred_train = modelo.predict(
        X_train_svm_reducido
    )

    pred_val = modelo.predict(
        X_val_svm
    )

    accuracy_train = accuracy_score(
        y_train_svm_reducido,
        pred_train
    )

    accuracy_val = accuracy_score(
        y_val,
        pred_val
    )

    resultados_svm.append({
        "Modelo": nombre,
        "C": valor_c,
        "Accuracy Train": accuracy_train,
        "Accuracy Validation": accuracy_val,
        "Tiempo (s)": tiempo
    })

    modelos_svm[nombre] = modelo

    print(f"Accuracy train: {accuracy_train:.4f}")
    print(f"Accuracy validation: {accuracy_val:.4f}")
    print(f"Tiempo: {tiempo:.2f} segundos")


Entrenando SVM C=0.01
Accuracy train: 0.9262
Accuracy validation: 0.5822
Tiempo: 957.98 segundos

Entrenando SVM C=0.1


C:\Users\belen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Accuracy train: 1.0000
Accuracy validation: 0.6125
Tiempo: 1806.95 segundos


In [13]:
tabla_resultados_svm = pd.DataFrame( resultados_svm )

tabla_resultados_svm = (
    tabla_resultados_svm
    .sort_values(  by="Accuracy Validation",  ascending = False )
    .reset_index(drop=True)
)

display(tabla_resultados_svm )

,Modelo,C,Accuracy Train,Accuracy Validation,Tiempo (s)
0,SVM C=0.1,0.10,1.0000,0.612495,1806.950245
1,SVM C=0.01,0.01,0.9262,0.582215,957.979855


In [14]:

mejor_nombre = tabla_resultados_svm.iloc[0]["Modelo"]
mejor_c = tabla_resultados_svm.iloc[0]["C"]

mejor_svm = modelos_svm[mejor_nombre]

print("Mejor configuración:")
print(f"Modelo: {mejor_nombre}")
print(f"C: {mejor_c}")

Mejor configuración:
Modelo: SVM C=0.1
C: 0.1


In [15]:
inicio_test = time.time()

y_pred_test_svm = mejor_svm.predict(
    X_test_svm
)

tiempo_test = time.time() - inicio_test

accuracy_test_svm = accuracy_score(
    y_test,
    y_pred_test_svm
)

print("\n" + "=" * 50)
print("RESULTADO FINAL DEL SVM")
print("=" * 50)

print(f"Mejor configuración: {mejor_nombre}")
print(f"Accuracy Test: {accuracy_test_svm:.4f}")
print(f"Tiempo de predicción: {tiempo_test:.2f} segundos")


RESULTADO FINAL DEL SVM
Mejor configuración: SVM C=0.1
Accuracy Test: 0.6198
Tiempo de predicción: 0.57 segundos


In [16]:
import tensorflow as tf
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    layers.RandomContrast(0.10)
], name="data_augmentation")